# VectoVecto Tier B training (DEBUG build — logs everything to run.log)

Previous run crashed silently. This version captures **all stdout/stderr to `/kaggle/working/run.log`** so we can diagnose from the downloaded output even on crash.

Required: **GPU T4 x2 + Internet ON + DIV2K dataset added as Input**.

In [ ]:
# Cell 1 — Clone repo + setup
import subprocess, sys, os, time

REPO_URL = 'https://github.com/DontHash/VectoVecto.git'
REPO_DIR = '/kaggle/working/VectoVecto'
LOG_PATH = '/kaggle/working/run.log'

if not os.path.isdir(REPO_DIR):
    print('Cloning repo...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Repo already at {REPO_DIR}')

os.chdir(REPO_DIR)
print('Repo cloned. Files in repo root:')
for f in sorted(os.listdir('.'))[:20]:
    print(' ', f)

# Open a log file that we'll write everything into
log = open(LOG_PATH, 'w', buffering=1)  # line-buffered
def logline(msg):
    log.write(str(msg) + '\n'); log.flush()
logline(f'=== VectoVecto Tier B training run — {time.strftime("%Y-%m-%d %H:%M:%S")} ===')
logline(f'Python: {sys.version}')
logline(f'CWD: {os.getcwd()}')
logline('---')
print('Log opened at', LOG_PATH)

In [ ]:
# Cell 2 — Install deps + fix PyTorch for P100/T4 compat
# Kaggle's preinstalled PyTorch (2.5+cu124) dropped sm_60 (P100) support.
# We detect the GPU compute capability and reinstall cu118 PyTorch if needed
# (cu118 supports sm_37..sm_90 = works on P100 AND T4).
import subprocess, sys
def run(cmd, logpath=log):
    logpath.write(f'$ {cmd}\n'); logpath.flush()
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    logpath.write(p.stdout); logpath.write(p.stderr); logpath.flush()
    return p.returncode, p.stdout, p.stderr

rc, out, err = run(f'{sys.executable} -m pip install -q tqdm')
logline('pip install tqdm done')

# Check GPU compute capability
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    logline(f'GPU: {name} (sm_{cap[0]}{cap[1]})')
    print(f'GPU: {name} (sm_{cap[0]}{cap[1]})')
    if cap[0] < 7:
        logline(f'GPU sm_{cap[0]}{cap[1]} < sm_70 -> reinstalling PyTorch cu118 for compat')
        print(f'sm_{cap[0]}{cap[1]} < sm_70 -> reinstalling PyTorch with cu118...')
        rc2, out2, err2 = run(f'{sys.executable} -m pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118 --force-reinstall')
        logline(f'cu118 install rc={rc2}')
        if rc2 != 0:
            logline(f'cu118 install FAILED:\n{err2}')
            print('FAILED to install cu118 PyTorch:', err2[-500:])
        else:
            logline('cu118 PyTorch installed OK')
            print('cu118 PyTorch installed. Reloading...')
else:
    logline('FATAL: GPU not enabled')
    log.close()
    raise RuntimeError('GPU not enabled. Settings -> Accelerator -> GPU.')

# Verify CUDA works after potential reinstall
rc, out, err = run(f'{sys.executable} -c "import torch; print(torch.__version__); x=torch.rand(2,2,device=\\\"cuda\\\"); print(x.sum().item()); print(\\\"CUDA OK\\\")"')
logline(f'CUDA verify: rc={rc}, out={out.strip()}, err={err.strip()[:200]}')
print('CUDA verify rc:', rc)
print(out)
if err:
    print('STDERR:', err[:500])
if rc != 0 or 'CUDA OK' not in out:
    logline('FATAL: CUDA still broken after reinstall')
    log.close()
    raise RuntimeError('CUDA not working. Check run.log for details.')
logline('GPU + CUDA verified OK')

In [ ]:
# Cell 3 — Locate DIV2K (logs every candidate path it tried)
import os
found_dir = None
for entry in os.listdir('/kaggle/input/'):
    base = os.path.join('/kaggle/input/', entry)
    logline(f'  /kaggle/input/{entry} exists={os.path.isdir(base)}')
    if os.path.isdir(base):
        for sub in ('DIV2K_train_HR', 'DIV2K_train_HR/DIV2K_train_HR', 'train_HR'):
            p = os.path.join(base, sub)
            if os.path.isdir(p):
                imgs = [f for f in os.listdir(p) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                logline(f'    {p} -> {len(imgs)} images')
                if imgs:
                    found_dir = p
                    break
        if found_dir: break

if found_dir is None:
    logline('FATAL: DIV2K HR not found')
    log.close()
    raise RuntimeError('DIV2K not found')
logline(f'DIV2K HR: {found_dir}')
logline(f'Images: {len(os.listdir(found_dir))}')
os.environ['DIV2K_HR_DIR'] = found_dir
out_dir = '/kaggle/working/deep_sr'
os.makedirs(out_dir, exist_ok=True)
os.environ['ARTIFACTS_DIR'] = out_dir
logline(f'ARTIFACTS_DIR={out_dir}')
print('DIV2K:', found_dir)

In [ ]:
# Cell 4 — Smoke test via train_deep_sr.py --smoke (logs every line)
import subprocess, sys, time
t0 = time.time()
logline('--- SMOKE TEST START ---')
p = subprocess.run([sys.executable, 'train_deep_sr.py', '--smoke', '--device', 'cuda', '--allow-cpu'],
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
log.write(p.stdout); log.write('\n'); log.flush()
logline(f'--- SMOKE TEST END (rc={p.returncode}, {time.time()-t0:.1f}s) ---')
print('Smoke rc:', p.returncode)
print(p.stdout[-1500:])

In [ ]:
# Cell 5 — TRAINING (adjust epochs below; checkpoints save every 2000 iters)
# v8: STRONGER sharpening weights (perc=0.5, adv=0.02 vs 0.1/0.005 before).
#     5-epoch test was stable + color-correct but BLURRY (GAN weight was ~zero).
# Each epoch = 250 iters. P100 runs ~0.85s/it -> 1 epoch ~3.5 min.
#   --epochs 5   -> 1250 iters, ~18 min  (sanity check)
#   --epochs 100 -> 25k iters, ~6h      (overnight target: sharp textures)
import subprocess, sys, time
t0 = time.time()
logline('--- TRAINING START ---')
p = subprocess.run([sys.executable, 'train_deep_sr.py',
                    '--device', 'cuda',
                    '--epochs', '100', '--batch', '8', '--patch', '32',
                    '--iters-per-epoch', '250', '--save-every', '2000',
                    '--log-every', '50', '--workers', '2',
                    '--w-perceptual', '0.5', '--w-adv', '0.02'],
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
log.write(p.stdout); log.write('\n'); log.flush()
logline(f'--- TRAINING END (rc={p.returncode}, {time.time()-t0:.1f}s) ---')
print('Training rc:', p.returncode)
print(p.stdout[-3000:])

In [ ]:
# Cell 6 — Close log + list outputs
log.flush(); log.close()
print('=== run.log saved to /kaggle/working/run.log ===')
print('=== /kaggle/working/ contents ===')
for f in os.listdir('/kaggle/working/'):
    p = os.path.join('/kaggle/working/', f)
    if os.path.isfile(p):
        size_kb = os.path.getsize(p) / 1024
        print(f'  {f}: {size_kb:.1f} KB')
    else:
        print(f'  {f}/')
print('=== artifacts/deep_sr/ ===')
if os.path.isdir('/kaggle/working/deep_sr'):
    for f in os.listdir('/kaggle/working/deep_sr'):
        size_mb = os.path.getsize(os.path.join('/kaggle/working/deep_sr', f)) / 1e6
        print(f'  {f}: {size_mb:.1f} MB')
else:
    print('  (no checkpoints — training crashed earlier')
print()
print('Download run.log from Output panel to see exactly what happened.')